In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix,accuracy_score,precision_score,
                             recall_score,f1_score,roc_auc_score)
from sklearn.base import clone
from sklearn.pipeline import Pipeline         
from sklearn.svm import SVC                    
from sklearn.ensemble import RandomForestClassifier

In [4]:
df=pd.read_csv(r"C:\Users\KSA\Desktop\5th-sem_ojt\project1_diabetes\data\prosessed\najaah_p1w2_features.csv")

feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_catogory","glucose_band"]


x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]


x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)


candidates = {
    "LogReg": ("scaled", LogisticRegression(max_iter=1000)),
    "Decision Tree": ("raw", DecisionTreeClassifier(max_depth=3, random_state=42)),
    "KNN": ("scaled", KNeighborsClassifier(n_neighbors=25)),
    "SVC balanced": ("scaled",SVC(class_weight="balanced", random_state=42)),
    "RF balanced": ("raw", RandomForestClassifier(
        max_depth=5, class_weight="balanced", random_state=42
    )),
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50   

print("rows:", len(df), "| train:", len(x_train), "| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("features after get_dummies:", x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)



rows: 950 | train: 760 | test: 190
diabetic in test set: 55
features after get_dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


In [5]:
df.columns

Index(['patient_id', 'age', 'gender', 'pregnancies', 'glucose',
       'blood_pressure', 'skin_thickness', 'insulin', 'bmi',
       'diabetes_pedigree', 'physical_activity', 'smoker', 'last_checkup_date',
       'outcome', 'age_band', 'bmi_catogory', 'glucose_band',
       'insulin_glucose_ratio', 'pregnancies_rate', 'since_checkup',
       'activity_smoker'],
      dtype='str')

# re split the data and watch the score moved


In [6]:
rows=[]

for rs in [0,1,7,42,99]:
    xa,xb,ya,yb = train_test_split(x,y,test_size=0.2,
    random_state=rs,stratify=y)
    
    row = {"split":rs}
    for name, (kind,model) in candidates.items():
        m=clone(model)
        if kind =="scaled":
            sc =StandardScaler().fit(xa)
            m.fit(sc.transform(xa),ya)
            pred = m.predict(sc.transform(xb))
        else:
            m.fit(xa,ya)
            pred =m.predict(xb)
        row[name]= round(recall_score(yb,pred),3)
    rows.append(row)      


splits = pd.DataFrame(rows).set_index("split")
print(splits)
spread = (splits.max() - splits.min()).round(3)
print("\nspread (best - worst):\n", spread.sort_values(ascending=False))    

        

       LogReg  Decision Tree    KNN  SVC balanced  RF balanced
split                                                         
0       0.218          0.091  0.145         0.600        0.618
1       0.255          0.382  0.164         0.745        0.600
7       0.255          0.182  0.200         0.636        0.582
42      0.291          0.291  0.200         0.618        0.673
99      0.273          0.236  0.200         0.709        0.655

spread (best - worst):
 Decision Tree    0.291
SVC balanced     0.145
RF balanced      0.091
LogReg           0.073
KNN              0.055
dtype: float64


**K-fold cross-validation** is a way to check how well a model may perform on new data.

1. Split the data into **K groups**, called folds.
2. Train the model using all but one fold.
3. Test it on the fold left out.
4. Repeat until each fold has been used for testing.
5. Average the scores from all rounds.

For example, with **5-fold cross-validation**, the model is trained and tested five times, each time using a different 20% of the data for testing.

# cross validation
**setup cross validation the right way**

In [7]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

for i, (tr, te) in enumerate(cv.split(x_train, y_train), 1):
    print(f"fold {i}: train {len(tr)} score {len(te)} "
          f"diabetic in score fold {y_train.iloc[te].sum()}")

def make_model(kind, model):
    """Scaled models get the scaler INSIDE a Pipeline, so it refits per fold."""
    if kind == "scaled":
        return Pipeline([("scale", StandardScaler()), ("model", model)])
    return model

print()
for name, (kind, model) in candidates.items():
    print(f"{name:15s} -> {type(make_model(kind, model)).__name__}")

fold 1: train 608 score 152 diabetic in score fold 43
fold 2: train 608 score 152 diabetic in score fold 43
fold 3: train 608 score 152 diabetic in score fold 44
fold 4: train 608 score 152 diabetic in score fold 44
fold 5: train 608 score 152 diabetic in score fold 44

LogReg          -> Pipeline
Decision Tree   -> DecisionTreeClassifier
KNN             -> Pipeline
SVC balanced    -> Pipeline
RF balanced     -> RandomForestClassifier


# **cross validate every candidate**

In [8]:
from sklearn.model_selection import cross_validate

rows = []
for name, (kind, model) in candidates.items():
  est = make_model(kind, model)
  res = cross_validate(
      est,x_train,y_train,cv=cv,scoring=["recall", "f1"],
  )
  single = clone(est).fit(x_train, y_train)
  rows.append({
      "model": name,
      "cv_recall": round(res["test_recall"].mean(), 3),
      "cv_std": round(res["test_recall"].std(), 3),
      "cv_f1": round(res["test_f1"].mean(), 3),
      "week6_recall":round(recall_score(y_test,single.predict(x_test)),3),
      "folds": np.round(res["test_recall"],3).tolist(),

  })
cv_table = pd.DataFrame(rows).sort_values("cv_recall",ascending=False)
print(cv_table.to_string(index=False))

        model  cv_recall  cv_std  cv_f1  week6_recall                               folds
 SVC balanced      0.716   0.025  0.559         0.618 [0.744, 0.744, 0.682, 0.705, 0.705]
  RF balanced      0.661   0.031  0.539         0.673 [0.721, 0.651, 0.659, 0.636, 0.636]
       LogReg      0.303   0.073  0.389         0.291  [0.419, 0.326, 0.318, 0.25, 0.205]
Decision Tree      0.216   0.076  0.296         0.291 [0.116, 0.326, 0.159, 0.205, 0.273]
          KNN      0.184   0.031  0.267         0.200 [0.209, 0.233, 0.159, 0.159, 0.159]


In [ ]:
cv_table["low"] = (cv_table["cv_recall"] - cv_table["cv_std"]).round(3)
cv_table["high"] = (cv_table["cv_recall"] + cv_table["cv_std"]).round(3)
cv_table["clears_target"] = cv_table["cv_recall"] >= TARGET_RECALL
cv_table["clears_guard"] = cv_table["cv_f1"] >= GUARD_F1

print(cv_table[["model","cv_recall","low","high","cv_f1",
                "clears_target","clears_guard"]].to_string(index=False))

        model  cv_recall   low  high  cv_f1  clears_target  clears_guard
 SVC balanced      0.716 0.691 0.741  0.559           True          True
  RF balanced      0.661 0.630 0.692  0.539          False          True
       LogReg      0.303 0.230 0.376  0.389          False         False
Decision Tree      0.216 0.140 0.292  0.296          False         False
          KNN      0.184 0.153 0.215  0.267          False         False


## Hyperparameter Tuning

Hyperparameter tuning finds the model settings that give the best results.

- **GridSearchCV** tests every combination of settings you provide. It is thorough but can be slow.
- **RandomizedSearchCV** tests a chosen number of random combinations. It is faster for large searches.

Both use cross-validation to compare settings across multiple data splits.

**Use GridSearchCV** for a small set of options.  
**Use RandomizedSearchCV** for many options or when you need a faster search.

In [31]:
param_grid ={
    "max_depth"      : [ 3,5,8,None],
    "min_samples_leaf": [1,5,10],
    "class_weight"  : ["balanced","balanced_subsample"]
    }

combos = 1

for values in param_grid.values():
    combos *= len(values)

folds = cv.get_n_splits()
total_fits = combos*folds

print('combination           :',combos)
print("cross-validtion folds :",folds)
print("total model fits      :",total_fits)

combination           : 24
cross-validtion folds : 5
total model fits      : 120


# Run the grid search on the leading forest

In [34]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

gs_recall = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1


)
gs_recall.fit(x_train,y_train)

print("Best setting   :",gs_recall.best_params_)
print("Best cv recall :",round(gs_recall.best_score_,3))

top = pd.DataFrame(gs_recall.cv_results_)

print(
    top[["params","mean_test_score","std_test_score","rank_test_score",]]
    .sort_values("rank_test_score")
    .head(5)
    .to_string(index=False)
)

Best setting   : {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}
Best cv recall : 0.707
                                                              params  mean_test_score  std_test_score  rank_test_score
{'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}         0.706554        0.035347                1
 {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}         0.692812        0.029575                2
 {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 5}         0.688161        0.033213                3
 {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 5}         0.688055        0.023213                4
{'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 10}         0.679281        0.040479                5
